<div style='background:#000000;border-radius:10px;padding:26px 30px;border-top:6px solid #3777FF;border-bottom:3px solid #0EE071;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'>
<p style='color:#0EE071;margin:0;letter-spacing:2px;font-size:12px'>NOVA IMS · ENTERPRISE DATA SCIENCE BOOTCAMP 2026 · PROJETO D</p>
<h1 style='color:#FFFFFF;margin:6px 0 4px 0'>Telco Customer Churn · <span style='color:#3777FF'>Retention Management</span></h1>
<p style='color:#FFFFFF;margin:0;font-size:15px'>Notebook 02 — Modelos · avaliação contra o benchmark IBM · priorização por valor</p>
</div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>Índice — Estrutura do notebook</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Pré-requisito: correr primeiro <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>01_preprocessing_eda.ipynb</code> ou <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>python scripts/run_preprocessing.py</code> (gera <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code>). O mesmo fluxo corre sem notebook com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>python scripts/run_modelling.py</code>; as decisões estão resumidas no <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>README.md</code>.</p>
<ul style='margin:6px 0 2px 0'><li>1 · Setup do ambiente</li><li>2 · Carregamento dos artefactos</li><li>3 · Os modelos e as métricas</li><li>4 · Tuning</li><li>5 · Comparação em validação cruzada e escolha do modelo</li><li>6 · Satisfaction Score, com e sem</li><li>7 · Avaliação final no holdout</li><li>8 · Interpretação</li><li>9 · Priorização dos clientes estabelecidos: P(churn) × CLTV</li><li>10 · Clientes novos: regra de onboarding</li><li>11 · Exportação</li><li>12 · Conclusões para o relatório</li></ul></div></div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>1 · Setup do ambiente</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Mesmo setup do notebook 01. Dois módulos do projeto fazem o trabalho desta fase:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling</code>: modelos, tuning, métricas, avaliação e priorização</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>model_plots</code>: gráficos da modelação, com a paleta do projeto</li></ul></div></div>

In [ ]:
# --- Bibliotecas ---
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# --- Encontrar a pasta do projeto (a que tem a pasta "src") ---
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# --- Módulos do próprio projeto (pasta src/telco_churn) ---
from telco_churn import features, model_plots, modelling, plots, retention, scoring
from telco_churn.config import ARTIFACTS_DIR, DATA_PROCESSED, TAB_DIR, ensure_dirs
from telco_churn.style import apply_theme, style_table

# --- Opções ---
warnings.filterwarnings("ignore")         # esconder avisos
pd.set_option("display.max_columns", 60)  # mostrar mais colunas nas tabelas

SHOW = True    # True = mostrar os gráficos (são sempre guardados em ficheiro)

ensure_dirs()
apply_theme()
print("Pasta do projeto:", ROOT)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>2 · Carregamento dos artefactos</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Carrega <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code> (listas de features e target) e os CSV de treino, teste e Joined. Os CSV são lidos com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>keep_default_na=False</code> para preservar o literal <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>None</code> em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Offer</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Internet Type</code>.</p></div></div>

In [ ]:
# Carregar os dados preparados no notebook 01
train, test, joined, cfg = modelling.load_processed()

NUM = cfg["features_num"]   # colunas numéricas
CAT = cfg["features_cat"]   # colunas categóricas
TARGET = cfg["target"]      # coluna que queremos prever (churn)
FEATURES = NUM + CAT        # todas as colunas usadas pelo modelo

# X = colunas de entrada, y = resposta (churn)
X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("Treino:", len(train), "clientes | taxa de churn:", round(y_train.mean(), 3))
print("Teste: ", len(test), "clientes | taxa de churn:", round(y_test.mean(), 3))
print("Joined:", len(joined), "clientes (sem resposta, só para prever)")
print(len(NUM), "numéricas +", len(CAT), "categóricas =", len(FEATURES), "features")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>3 · Os modelos e as métricas</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Decidimos comparar <b style='color:#3777FF'>quatro modelos</b>, do mais simples ao mais complexo, e <b style='color:#3777FF'>três referências</b> que nunca são escolhidas:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Logistic Regression</b>: o modelo linear, simples e interpretável</li><li><b style='color:#3777FF'>Decision Tree</b>: perguntas sim/não, fácil de explicar</li><li><b style='color:#3777FF'>Random Forest</b> (bagging) e <b style='color:#3777FF'>Gradient Boosting</b> (boosting): ensembles de árvores, normalmente os melhores em dados tabulares</li><li>Referências: a <b style='color:#3777FF'>Dummy</b> (dá a todos a taxa de churn), uma <b style='color:#3777FF'>regra de negócio</b> sem ML (1 ponto por fator de risco: Month-to-Month, tenure ≤ 12 meses, fibra ótica) e o <b style='color:#3777FF'>Churn Score IBM</b></li></ul>
<p>A regressão logística usa o pré-processador <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>linear</code> (com escala); as árvores usam o <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>trees</code> (sem escala), porque comparam cada variável consigo própria.</p></div></div>

In [ ]:
# Família e funcionamento de cada modelo
display(style_table(modelling.MODEL_INFO))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>3.1 · As nove métricas</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Decidimos avaliar cada modelo com as mesmas nove métricas. As quatro primeiras usam as probabilidades; as cinco últimas precisam de uma decisão sim/não e usam o limiar 0,5.</p>
<p>A <b style='color:#3777FF'>ROC AUC</b> é a métrica de seleção, porque a equipa de retenção trabalha com uma lista ordenada. O <b style='color:#3777FF'>Brier</b> mede a calibração, de que precisamos porque P(churn) multiplica o CLTV na priorização.</p></div></div>

In [ ]:
display(style_table(modelling.metrics_glossary()))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>4 · Tuning</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Cada modelo é um <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Pipeline([("prep", ...), ("clf", ...)])</code>: o pré-processador é ajustado dentro de cada fold, por isso não há fuga de informação na validação cruzada.</p>
<p>Usamos <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>GridSearchCV</code> com uma grelha pequena (2 ou 3 valores para 1 ou 2 hiperparâmetros por modelo), <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>StratifiedKFold</code> com 5 folds e ROC AUC, só no treino. O holdout não é usado até à secção 7. As grelhas estão em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling.BASE_MODELS</code>.</p></div></div>

In [ ]:
# Melhores hiperparâmetros de cada modelo (validação cruzada com 5 folds, só no treino)
best_params, tuning = modelling.tune_base_models(X_train, y_train, NUM, CAT)
tuning.to_csv(TAB_DIR / "10_tuning_modelos.csv")
display(style_table(tuning.sort_values("cv_auc", ascending=False), precision=4, highlight="cv_auc"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>5 · Comparação em validação cruzada e escolha do modelo</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Todos os modelos e referências são avaliados no <b style='color:#3777FF'>mesmo 5-fold</b> do treino com as nove métricas. Para cada modelo guardamos a média e o desvio-padrão em validação, a média no treino (para medir o sobreajuste) e as previsões *out-of-fold*: cada cliente recebe a probabilidade de um modelo que não o viu. Os gráficos desta secção usam essas previsões, sem tocar no holdout.</p>
<p>Nota sobre o código: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling.cv_evaluate</code> usa <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>joblib.Parallel</code> para treinar os 5 folds ao mesmo tempo. O resultado é igual ao de um ciclo <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>for</code> sobre os folds; só demora menos.</p></div></div>

In [ ]:
# Duas linhas de base (Dummy e regra de negócio) + 4 modelos afinados
models = modelling.build_models(NUM, CAT, best_params)

# Todos no mesmo 5-fold: nove métricas e previsões out-of-fold.
# O Churn Score IBM entra como referência e é avaliado exatamente nos mesmos folds.
ibm = {modelling.BENCHMARK: train["Churn Score"].to_numpy() / 100}
cv_table, oof = modelling.cv_evaluate(models, X_train, y_train, benchmarks=ibm)
cv_table.to_csv(TAB_DIR / "11_cv_modelos.csv")

cv_view = modelling.metrics_view(cv_table)            # médias em CV (modelos x métricas)
cv_train = modelling.metrics_view(cv_table, "train")  # médias no treino
cv_std = cv_table[[m + "_std" for m in modelling.METRICS]]
cv_std.columns = list(modelling.METRICS)
display(style_table(cv_view, precision=3, highlight="auc"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.1 · ROC AUC e escolha do modelo final</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>A barra de erro é o desvio-padrão entre folds. A barra preta é o Churn Score IBM (nos mesmos folds) e a cinzenta é a regra de negócio.</p>
<p>Regra de escolha: decidimos ficar com o modelo de <b style='color:#3777FF'>maior ROC AUC em CV</b>. Se a Logistic Regression ficar a menos de 0,01 dele, escolhemos a Logistic Regression, por ser mais fácil de explicar ao Head of Customer Retention.</p></div></div>

In [ ]:
model_plots.plot_cv_comparison(cv_table, show=SHOW)

BEST, motivo = modelling.select_model(cv_table)
pd.Series({"modelo_escolhido": BEST, "justificacao": motivo}).to_csv(
    TAB_DIR / "11d_selecao_modelo.csv", header=False)
print("Modelo escolhido:", BEST, "| AUC", round(cv_table.loc[BEST, "auc"], 4),
      "±", round(cv_table.loc[BEST, "auc_std"], 4))
print("Porquê:", motivo)
print("Referências em CV - AUC IBM:", round(cv_table.loc[modelling.BENCHMARK, "auc"], 3),
      "| regra de negócio:", round(cv_table.loc[modelling.RULE_BASELINE, "auc"], 3))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.3 · As nove métricas, modelo a modelo</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um painel por métrica e uma barra por modelo (média em CV, com o desvio-padrão). O <b style='color:#3777FF'>losango</b> é o valor no treino. Modelo final a <b style='color:#0EE071'>verde</b>, restantes modelos a <b style='color:#3777FF'>azul</b>, linha de base a cinzento.</p>
<p>Como ler: a linha de base tem accuracy alta e precision/recall a zero, porque nunca sinaliza ninguém. Nas métricas com limiar 0,5, a maioria dos modelos tem precision acima do recall: com 28% de churn, o limiar 0,5 é exigente (ver 5.6).</p></div></div>

In [ ]:
model_plots.plot_metrics_panel(cv_view, std=cv_std, train=cv_train, best=BEST, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.3 · Sobreajuste: treino vs validação</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>ROC AUC no treino e em validação cruzada. Quanto maior a distância (Δ), mais o modelo decorou o treino. As florestas e o boosting têm sempre algum Δ, porque cada árvore aprende bem o seu treino; o que importa é o valor em CV, que é o que se espera em clientes novos.</p></div></div>

In [ ]:
model_plots.plot_overfitting(cv_table.drop(index=modelling.REFERENCES), show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.4 · Curvas ROC e Precision-Recall</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Curvas de todos os modelos com as previsões out-of-fold, mais o Churn Score IBM a tracejado preto. O modelo final está a <b style='color:#0EE071'>verde e mais grosso</b>.</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>ROC</b>: % de churners apanhados (vertical) contra % de clientes fiéis sinalizados por engano (horizontal), em todos os limiares</li><li><b style='color:#3777FF'>Precision-Recall</b>: dos clientes sinalizados, quantos saíram de facto, à medida que se apanham mais churners. A linha pontilhada é a taxa de churn, o que se consegue ao acaso</li></ul></div></div>

In [ ]:
# oof já inclui o Churn Score IBM (benchmark) e a regra de negócio
model_plots.plot_roc_pr(y_train, oof, best=BEST, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.5 · Matrizes de confusão</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Para cada modelo, no limiar 0,5: nº de clientes e % da classe real. A diagonal são os acertos. O quadrado de <b style='color:#3777FF'>baixo à esquerda</b> são os churners que o modelo não detetou, o erro mais caro para a retenção; o de cima à direita são clientes fiéis contactados sem necessidade.</p></div></div>

In [ ]:
model_plots.plot_confusion_matrices(y_train, oof, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.6 · Limiar de decisão e capacidade da equipa</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O modelo dá uma probabilidade; para agir é preciso um limiar. Baixar o limiar apanha mais churners (o recall sobe), mas contacta mais clientes que ficariam (a precision desce).</p>
<p>Decidimos usar dois limiares:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>0,5</b>, o limiar por omissão das métricas;</li><li><b style='color:#3777FF'>top 20% dos clientes estabelecidos</b>: a P(churn) a partir da qual um cliente com mais de 3 meses de casa está entre os 20% de maior risco. Representa a capacidade da equipa de retenção. É fixado aqui, nas previsões *out-of-fold*, e aplicado depois ao holdout e à app. Usamos só os clientes estabelecidos porque os clientes novos têm P ≈ 1 por um efeito do registo (secção 7.1) e ocupariam o top 20%.</li></ul>
<p>O nº de clientes a contactar sai da análise custo-benefício (secção 9.2), não de uma métrica.</p></div></div>

In [ ]:
limiares = modelling.threshold_table(y_train, oof[BEST])
limiares.to_csv(TAB_DIR / "11b_limiares_modelo_final.csv")
display(style_table(limiares, precision=3))

# Capacidade da equipa: top 20% dos clientes estabelecidos (tenure > 3), nas previsões OOF
est_train = features.is_established(train).to_numpy()
CAPACITY_CUT = modelling.capacity_threshold(oof[BEST][est_train])
LIMIARES = {"0,5 (por omissão)": 0.5, "top 20% dos estabelecidos": CAPACITY_CUT}
limiares_cv = modelling.decision_table(y_train[est_train], oof[BEST][est_train], LIMIARES)
limiares_cv.to_csv(TAB_DIR / "11e_limiares_cv.csv")
display(style_table(limiares_cv, precision=3))
model_plots.plot_threshold(y_train, oof[BEST], BEST, marker=CAPACITY_CUT, show=SHOW)
print("P(churn) do top 20% dos clientes estabelecidos:", round(CAPACITY_CUT, 3))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>6 · Satisfaction Score — com e sem</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>No notebook 01, <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> 1–2 corresponde a 100% de churn: suspeita de ser registada no contexto da saída.
Decidimos treinar o modelo escolhido <b style='color:#0EE071'>com e sem</b> a variável, no mesmo 5-fold, e documentamos a diferença.</p>
<p>Se o ganho for grande, é sinal de leakage: a versão com a variável <b style='color:#3777FF'>não serve para retenção preventiva</b>, porque o score só existe tarde demais.</p></div></div>

In [ ]:
# Modelo A: SEM Satisfaction Score (é o modelo escolhido na secção 5; o CV já foi feito)
cv_sem = cv_table.loc[[BEST], modelling.SUMMARY_COLS]   # AUC (± desvio), PR-AUC e Brier
cv_sem = cv_sem.rename(index={BEST: BEST + " — sem Satisfaction"})

# Modelo B: COM Satisfaction Score (igual, mas com mais uma coluna numérica)
NUM_SAT = NUM + ["Satisfaction Score"]
FEATURES_SAT = NUM_SAT + CAT
modelo_com = modelling.build_models(NUM_SAT, CAT, best_params, baseline=False)[BEST]
cv_com, _ = modelling.cv_evaluate({BEST + " — com Satisfaction": modelo_com}, train[FEATURES_SAT], y_train)
cv_com = cv_com[modelling.SUMMARY_COLS]

# Juntar as duas linhas numa só tabela
sat_cv = pd.concat([cv_sem, cv_com])
sat_cv.to_csv(TAB_DIR / "12_variante_satisfaction.csv")
display(style_table(sat_cv, precision=4, highlight="auc"))

auc_sem = cv_sem["auc"].iloc[0]   # iloc[0] = primeira (e única) linha
auc_com = cv_com["auc"].iloc[0]
print("Diferença de AUC (com - sem):", round(auc_com - auc_sem, 4))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>7 · Avaliação final no holdout</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>O holdout (20%) é usado uma única vez, depois de todas as escolhas feitas: todos os modelos são treinados no treino completo e avaliados no teste. O modelo final já foi escolhido na secção 5; os restantes aparecem só para transparência.</p>
<p>O Churn Score IBM é avaliado no mesmo holdout. O seu Brier é só uma referência, porque Churn Score / 100 não é uma probabilidade calibrada. A versão com Satisfaction Score aparece à parte, para mostrar o efeito do leakage.</p></div></div>

In [ ]:
final_model = models[BEST]

# Treinar cada modelo no treino completo e prever P(churn) no teste
probas = modelling.fit_predict(models, X_train, y_train, X_test)

# Versão com Satisfaction Score (só para mostrar o efeito do leakage)
probas[BEST + " + Satisfaction (leakage)"] = modelling.fit_predict(
    {"sat": modelo_com}, train[FEATURES_SAT], y_train, test[FEATURES_SAT])["sat"]

# Tabela de métricas no teste (inclui o Churn Score IBM)
holdout = modelling.holdout_report(probas, y_test, test["Churn Score"])
holdout.to_csv(TAB_DIR / "13_holdout.csv")
display(style_table(holdout, precision=4, highlight="auc"))

p_test = probas[BEST]   # P(churn) do modelo final para cada cliente do teste

# Curva ROC e calibração
model_plots.plot_roc_calibration(y_test, {
    BEST: p_test,
    "Churn Score IBM": test["Churn Score"] / 100,
    "Logistic Regression": probas["Logistic Regression"],
}, show=SHOW)
model_plots.plot_metrics_panel(holdout.drop(index=modelling.BASELINE, errors="ignore"), best=BEST,
                               name="24b_painel_metricas_holdout",
                               title="Nove métricas por modelo — holdout (20%)", show=SHOW)

# Calibração: por faixa de P(churn), a P média e a taxa de churn real
calibracao = modelling.calibration_table(y_test, p_test)
calibracao.to_csv(TAB_DIR / "13c_calibracao_holdout.csv")
display(style_table(calibracao, precision=3, highlight="churn_real"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>7.1 · Clientes novos e clientes estabelecidos</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>No notebook 01 (secção 6.1) vimos que, dentro de Churned + Stayed, <b style='color:#3777FF'>todos os clientes com tenure ≤ 3 meses saíram</b>. Não é comportamento: os clientes novos que ficaram estão rotulados Joined e ficam fora da população. O modelo aprende por isso que "cliente recente = churn", e uma parte do seu AUC vem desses casos fáceis.</p>
<p>Decidimos manter um só modelo, treinado com a população do enunciado, e usar a P(churn) <b style='color:#3777FF'>só nos clientes estabelecidos</b> (tenure > 3). Os clientes novos seguem uma regra de onboarding (secção 10).</p>
<p>Para confirmar que o modelo serve nos estabelecidos, avaliamo-lo só nesses clientes do holdout, ao lado do Churn Score IBM, e verificamos se continua calibrado. É esta calibração que permite usar P(churn) × CLTV.</p></div></div>

In [ ]:
est = features.is_established(test).to_numpy()   # True = tenure > 3
test_est = test[est]
y_test_est = y_test[est]
p_test_est = p_test[est]
print("Clientes estabelecidos no holdout:", len(test_est),
      "| taxa de churn:", round(y_test_est.mean(), 3))

holdout_est = modelling.holdout_report({BEST: p_test_est}, y_test_est, test_est["Churn Score"])
holdout_est.to_csv(TAB_DIR / "13d_holdout_estabelecidos.csv")
display(style_table(holdout_est, precision=4, highlight="auc"))

# Os limiares fixados na secção 5.6, aplicados uma vez ao holdout
limiares_holdout = modelling.decision_table(y_test_est, p_test_est, LIMIARES)
limiares_holdout.to_csv(TAB_DIR / "13b_limiares_holdout_estabelecidos.csv")
display(style_table(limiares_holdout, precision=3))

calibracao_est = modelling.calibration_table(y_test_est, p_test_est)
calibracao_est.to_csv(TAB_DIR / "13e_calibracao_estabelecidos.csv")
display(style_table(calibracao_est, precision=3, highlight="churn_real"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>8 · Interpretação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><b style='color:#0EE071'>Permutation importance</b> no holdout: quanto cai o AUC quando os valores de uma variável são baralhados.
Funciona para qualquer modelo e ao nível das variáveis originais, não das colunas one-hot — mais útil para explicar ao Head of Customer Retention.</p>
<p>Barras <b style='color:#3777FF'>azuis</b> = numéricas, <b style='color:#0EE071'>verdes</b> = categóricas.</p></div></div>

In [ ]:
# Quanto cai o AUC quando baralhamos os valores de cada coluna?
# (5 repetições; valores negativos passam a 0; da mais para a menos importante)
importance = modelling.permutation_ranking(final_model, X_test, y_test)
importance.to_frame("queda_auc").to_csv(TAB_DIR / "14_importancia_permutacao.csv")

titulo = "Permutation importance — " + BEST + " (queda de AUC, holdout)"
plots.plot_ranking(importance.head(20), titulo, "27_importancia_permutacao",
                   highlight=set(CAT), show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>9 · Priorização dos clientes estabelecidos: P(churn) × CLTV</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O objetivo não é só prever churn, é decidir <b style='color:#3777FF'>quem contactar primeiro</b>. Cada cliente recebe uma
<b style='color:#0EE071'>perda esperada = P(churn) × CLTV</b> e a lista de intervenção é ordenada por esse valor.</p>
<p>Usamos o modelo nos clientes estabelecidos do holdout (secção 7.1). A <b style='color:#0EE071'>curva de captura</b> mede, que % do CLTV dos clientes que de facto saíram fica coberta ao contactar os top-k% da lista.
Compara quatro ordenações: o nosso P × CLTV, só o nosso P(churn), o Churn Score IBM × CLTV e o Churn Score IBM sozinho.</p></div></div>

In [ ]:
# 4 formas diferentes de ordenar os clientes estabelecidos do teste
orderings = {
    "Modelo: P × CLTV": p_test_est * test_est["CLTV"],
    "Modelo: só P(churn)": p_test_est,
    "IBM: Churn Score × CLTV": test_est["Churn Score"] * test_est["CLTV"],
    "IBM: só Churn Score": test_est["Churn Score"],
}

# Que % do CLTV perdido apanhamos ao contactar os primeiros k% de cada lista?
curve = modelling.capture_curve(y_test_est, test_est["CLTV"], orderings)
capture = modelling.capture_at(curve, cuts=(0.1, 0.2, 0.3))
capture.to_csv(TAB_DIR / "15_captura_valor.csv")
display(style_table(capture, precision=3))
model_plots.plot_capture_curve(curve, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.1 · Só P(churn) ou P(churn) × CLTV?</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>As duas ordenações respondem a perguntas diferentes: <b style='color:#3777FF'>só P(churn)</b> contacta primeiro quem tem mais risco; <b style='color:#0EE071'>P × CLTV</b> contacta primeiro quem tem maior perda esperada. Para o mesmo nº de contactos comparamos:</p>
<ul style='margin:6px 0 2px 0'><li>quantos clientes as duas listas têm <b style='color:#3777FF'>em comum</b> (se for perto de 100%, a escolha quase não importa);</li><li>que % dos churners cada lista apanha;</li><li>que % do CLTV perdido cada lista cobre (o objetivo da retenção).</li></ul>
<p>A comparação com o IBM é feita da mesma forma: P × CLTV com Churn Score × CLTV, e só P com só Churn Score.</p></div></div>

In [ ]:
orderings_compared = retention.compare_orderings(
    y_test_est, test_est["CLTV"],
    score_p=p_test_est,
    score_value=p_test_est * test_est["CLTV"],
)
orderings_compared.to_csv(TAB_DIR / "15b_P_vs_PxCLTV.csv")
display(style_table(orderings_compared, precision=3, highlight="em_comum"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.2 · Quantos clientes contactar? Análise custo-benefício</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>A lista diz por que ordem contactar. Para decidir quantos, é preciso comparar o valor salvo com o custo da campanha. Isso exige dois números que o dataset não tem:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>taxa_sucesso</code>: % dos churners contactados que ficam por causa do contacto</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>custo_contacto</code>: custo de contactar um cliente, incluindo o incentivo, na mesma unidade do CLTV. Paga-se por todos os contactados, mesmo pelos que ficariam de qualquer forma</li></ul>
<p>São pressupostos, por isso ficam em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>retention.CENARIOS</code>, em três cenários. Os valores são ilustrativos e devem ser substituídos pelos da empresa.</p>
<p>Regra que decidimos usar: <b style='color:#3777FF'>contactar quem tem P(churn) × CLTV × taxa_sucesso > custo_contacto</b>. Só usa o modelo, por isso aplica-se a uma lista futura, sem conhecer o churn. A tabela mostra também o valor líquido que essa decisão teria dado no holdout.</p></div></div>

In [ ]:
# ==========================================================================
# PARÂMETROS DE NEGÓCIO — PRESSUPOSTOS, NÃO DADOS
# Definidos uma só vez em src/telco_churn/retention.py (CENARIOS), para o notebook e o
# script usarem os mesmos valores. Substituir lá pelos valores reais da empresa.
#   taxa_sucesso   : % dos churners contactados que ficam por causa do contacto
#   custo_contacto : custo por cliente contactado (contacto + incentivo), na unidade do CLTV
# ==========================================================================
CENARIOS = retention.CENARIOS
display(style_table(pd.DataFrame(CENARIOS).T, precision=2))

In [ ]:
scenarios = retention.scenario_table(y_test_est, p_test_est, test_est["CLTV"], CENARIOS)
scenarios.to_csv(TAB_DIR / "20_cenarios_custo_beneficio.csv")
display(style_table(scenarios, precision=2))

# Curva de valor líquido de cada cenário (lista ordenada por P × CLTV)
value_curves = {}
for name, params in CENARIOS.items():
    value_curves[name] = retention.net_value_curve(
        y_test_est, test_est["CLTV"], p_test_est * test_est["CLTV"],
        params["taxa_sucesso"], params["custo_contacto"])
model_plots.plot_net_value(value_curves, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.3 · Lista de intervenção</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Lista ordenada dos clientes estabelecidos do holdout, onde podemos verificar o resultado. O mapa mostra onde ficam os top-N: o P × CLTV troca alguns clientes de risco alto mas pouco valor por clientes de valor alto com risco moderado.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TOP_N</code> é o orçamento de contactos — ajustar ao que a equipa de retenção consegue tratar.</p>
<p>Os <b style='color:#3777FF'>Joined</b> já não recebem P(churn): seria extrapolar um modelo que aprendeu o artefacto do tenure. São tratados na secção 10.</p></div></div>

In [ ]:
TOP_N = 200   # nº de clientes que a equipa de retenção consegue contactar

# Lista de prioridade dos clientes estabelecidos do teste + churn real (para verificar)
prio_test = modelling.priority_list(test_est, p_test_est)
churn_real = test_est[["Customer ID", TARGET]]
prio_test = prio_test.merge(churn_real, on="Customer ID")
model_plots.plot_priority_map(prio_test, TOP_N, show=SHOW)

# Resumo por decil (decil 1 = 10% de clientes com maior perda esperada)
grupos = prio_test.groupby("decil")
by_decile = pd.DataFrame({
    "clientes": grupos.size(),
    "p_churn_media": grupos["p_churn"].mean(),
    "churn_real": grupos[TARGET].mean(),
    "cltv_medio": grupos["CLTV"].mean(),
    "perda_esperada": grupos["perda_esperada"].sum(),
})
display(style_table(by_decile, precision=3, highlight="churn_real"))
display(style_table(prio_test.head(15), precision=3))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.4 · Que ação propor? Hipóteses por segmento</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Cada cliente da lista recebe uma <b style='color:#3777FF'>ação sugerida</b>, com regras simples sobre variáveis que a empresa conhece antes do churn (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>retention.suggest_action</code>). As regras inspiram-se na EDA e na importância das variáveis:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Proposta de contrato anual</b>: clientes Month-to-Month, onde se concentra o churn</li><li><b style='color:#3777FF'>Oferta de suporte técnico e segurança</b>: clientes com fibra sem Online Security nem Premium Tech Support</li><li><b style='color:#3777FF'>Revisão de tarifário antes da renovação</b>: restantes, que já têm contrato de 1 ou 2 anos</li></ul>
<p><b style='color:#0EE071'>Atenção: são hipóteses a testar, não efeitos demonstrados.</b> A importância das variáveis mostra que o modelo <i>usa</i> o tipo de contrato para ordenar clientes; não prova que mudar o contrato de um cliente o faça ficar. O churn real por segmento (abaixo) é descritivo.</p>
<p><b style='color:#3777FF'>Nota sobre o código:</b> as regras usam <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>np.select(condições, valores, default)</code>, que testa as condições por ordem e atribui a cada cliente o valor da primeira condição verdadeira; quem não cumpre nenhuma recebe o default. É o equivalente a uma sequência de if / elif / else aplicada a todas as linhas.</p>
<p>Para medir o efeito de cada ação: em cada segmento, sortear uma parte dos clientes (por exemplo 20%) como <b style='color:#3777FF'>grupo de controlo</b>, que não é contactado. Ao fim de um trimestre, a diferença de churn entre contactados e controlo estima o efeito da ação, e é esse valor que deve substituir a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>taxa_sucesso</code> assumida na secção 9.2.</p></div></div>

In [ ]:
# Juntar à lista as variáveis usadas nas regras e atribuir a ação sugerida
prio_test = prio_test.merge(test_est[["Customer ID"] + retention.ACTION_COLS], on="Customer ID")
prio_test["acao_sugerida"] = retention.suggest_action(prio_test)

# Orçamento: o N da regra do valor esperado no cenário Base (secção 9.2)
N_CAMPANHA = int(scenarios.loc["Base", "N_a_contactar"])
actions = retention.actions_summary(prio_test, N_CAMPANHA, TARGET)
actions.to_csv(TAB_DIR / "21_acoes_sugeridas.csv")
print("Top", N_CAMPANHA, "da lista (cenário Base), por ação sugerida:")
display(style_table(actions, precision=2, highlight="clientes"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>10 · Clientes novos: regra de onboarding</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Os Joined (454 clientes, 1 a 3 meses de casa) não recebem P(churn). O modelo daria a quase todos P ≈ 1, porque na população de treino todos os clientes novos saíram (secção 7.1).</p>
<p>Em vez disso, decidimos usar uma <b style='color:#3777FF'>regra de onboarding</b> tirada da EDA. Comparamos os clientes novos que saíram (Churned com tenure ≤ 3) com os que ficaram (Joined) e procuramos as variáveis onde a taxa de saída mais muda.</p>
<p>É uma hipótese de negócio, não um resultado validado. A taxa de saída é descritiva: depende de como o snapshot rotula os clientes e não é uma probabilidade. Para a validar, a regra deve ser aplicada a uma parte dos novos clientes, mantendo outra parte como grupo de controlo, e comparar a saída nos dois grupos ao fim de um trimestre.</p></div></div>

In [ ]:
# Base completa, com Customer Status, para juntar os novos que saíram e os que ficaram
merged = pd.read_csv(DATA_PROCESSED / "telco_merged_full.csv", keep_default_na=False, na_values=[""])
novos = merged[~features.is_established(merged)].copy()
novos["saiu"] = (novos["Customer Status"] == "Churned").astype(int)
print("Clientes novos (tenure <= 3):", len(novos))
display(style_table(novos["Customer Status"].value_counts().to_frame("clientes")))

# Taxa de saída dos clientes novos por contrato e por tipo de internet
for col in ["Contract", "Internet Type"]:
    by_col = novos.groupby(col)["saiu"].agg(clientes="size", taxa_saida="mean")
    display(style_table(by_col, precision=2, highlight="taxa_saida"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>10.1 · A regra e a lista de onboarding</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>A saída dos clientes novos concentra-se nos contratos <b style='color:#3777FF'>Month-to-Month</b> e, dentro destes, na <b style='color:#3777FF'>fibra ótica</b>. Daqui saem três segmentos, definidos em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>retention.onboarding_segment</code>:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>Onboarding prioritário</b>: Month-to-Month com Fiber Optic — por exemplo, contacto de boas-vindas e verificação da qualidade do serviço no 1.º mês</li><li><b style='color:#3777FF'>Onboarding standard</b>: restantes Month-to-Month — por exemplo, comunicação automática com as vantagens de um contrato anual</li><li>Sem ação específica: contratos de 1 ou 2 anos, onde a saída de clientes novos é rara</li></ul>
<p>As ações são exemplos para discutir com o Head of Customer Retention. Dentro de cada segmento, a lista ordena por CLTV.</p></div></div>

In [ ]:
# Segmento de cada cliente novo, taxa de saída por segmento e nº de Joined em cada um
novos["onboarding"] = retention.onboarding_segment(novos)
regra = retention.onboarding_rule(novos)
regra.to_csv(TAB_DIR / "19_regra_onboarding.csv")
display(style_table(regra, precision=2, highlight="taxa_saida"))

# Lista de onboarding: os Joined, por segmento e CLTV
onboarding_joined = retention.onboarding_list(joined)
display(style_table(onboarding_joined.head(15), precision=0))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>11 · Exportação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Ficheiros gerados (ignorados pelo Git):</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/artifacts/final_model.joblib</code>: modelo final treinado, lista de features, métricas e o corte do top 20%</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/prioridade_holdout.csv</code>: lista de intervenção dos clientes estabelecidos do holdout</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/onboarding_joined.csv</code>: lista de onboarding dos Joined, por segmento</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/tables/10_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>21_*</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/figures/21_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>30_*</code>: tabelas e gráficos desta fase (lista no <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>README.md</code>)</li></ul></div></div>

In [ ]:
# Guardar o modelo final e a informação de que a app precisa para o usar
inputs = [c for c in FEATURES if c not in features.NEW_NUM + features.NEW_CAT]
modelling.save_bundle(ARTIFACTS_DIR / "final_model.joblib", final_model, BEST, FEATURES, TARGET,
                      best_params[BEST], cv=cv_table.loc[BEST].to_dict(),
                      holdout=holdout.loc[BEST].to_dict(), selection=motivo,
                      benchmark_cv=cv_table.loc[modelling.BENCHMARK].to_dict(),
                      benchmark_holdout=holdout.loc[modelling.BENCHMARK].to_dict(),
                      holdout_estabelecidos=holdout_est.loc[BEST].to_dict(),
                      benchmark_holdout_estabelecidos=holdout_est.loc[modelling.BENCHMARK].to_dict(),
                      thresholds=LIMIARES, capacity_cut=CAPACITY_CUT,
                      form=scoring.form_spec(train, inputs),
                      cltv_median=float(train["CLTV"].median()))

# Guardar a lista de prioridade (estabelecidos) e a de onboarding (Joined)
prio_test.to_csv(DATA_PROCESSED / "prioridade_holdout.csv", index=False)
onboarding_joined.to_csv(DATA_PROCESSED / "onboarding_joined.csv", index=False)

print("Guardado: outputs/artifacts/final_model.joblib")
print("Guardado: data/processed/prioridade_holdout.csv")
print("Guardado: data/processed/onboarding_joined.csv")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>12 · Conclusões para o relatório</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>A célula seguinte resume os números-chave a partir das tabelas acima, para transcrever para o relatório:</p>
<ul style='margin:6px 0 2px 0'><li>modelo escolhido e porquê</li><li>desempenho do modelo final vs Churn Score IBM no mesmo holdout, em todos os clientes e só nos estabelecidos</li><li>efeito do Satisfaction Score (leakage)</li><li>valor captado com a priorização P(churn) × CLTV</li></ul></div></div>

In [ ]:
# Números-chave para o relatório
print("Modelo final:", BEST)
print("  Escolha:", motivo)
print("  Regra de negócio (sem ML): AUC teste", round(holdout.loc[modelling.RULE_BASELINE, "auc"], 3))
for nome, tabela in [("Todos os clientes", holdout), ("Só estabelecidos (tenure > 3)", holdout_est)]:
    m, ibm = tabela.loc[BEST], tabela.loc[modelling.BENCHMARK]
    print(nome + ":")
    print("  AUC    modelo", round(m["auc"], 3), "| IBM", round(ibm["auc"], 3))
    print("  PR-AUC modelo", round(m["pr_auc"], 3), "| IBM", round(ibm["pr_auc"], 3))
    print("  Brier  modelo", round(m["brier"], 3), "| IBM", round(ibm["brier"], 3), "(IBM só indicativo)")

auc_sat = holdout.loc[BEST + " + Satisfaction (leakage)", "auc"]
print("Satisfaction Score: AUC com a variável =", round(auc_sat, 3), "(fora do modelo final)")

# Comparar sempre ordenações do mesmo tipo: P × CLTV com P × CLTV, só P com só P
print("Contactando 20% dos clientes estabelecidos (% do CLTV perdido coberto):")
print("  P × CLTV: modelo", round(capture.loc["Modelo: P × CLTV", "top 20%"] * 100, 1),
      "% | IBM", round(capture.loc["IBM: Churn Score × CLTV", "top 20%"] * 100, 1), "%")
print("  Só P:     modelo", round(capture.loc["Modelo: só P(churn)", "top 20%"] * 100, 1),
      "% | IBM", round(capture.loc["IBM: só Churn Score", "top 20%"] * 100, 1), "%")
print("  Clientes em comum entre a lista por P e por P × CLTV:",
      round(orderings_compared.loc["top 20%", "em_comum"] * 100, 1), "%")
print("Campanha no cenário Base:", N_CAMPANHA, "clientes a contactar")